# Bakehouse Transactions

**Dataset:** `samples.bakehouse.sales_transactions`

**Difficulty:** Easy

**Topics:** aggregation, distinct, date, groupBy, F.min, F.max

In [0]:
from pyspark.sql import functions as F, types as T

## Learn — Aggregations and Date Extraction

| Function | What it does |
|----------|-------------|
| `F.sum("col")` | Sums all values in a column |
| `F.count("*")` | Counts all rows (including nulls) |
| `F.countDistinct("col")` | Counts unique non-null values |
| `F.min("col")` | Returns the smallest value — works on dates and timestamps too |
| `F.max("col")` | Returns the largest value — gives the most recent date/timestamp |
| `F.year(col)` | Extracts the year from a date/timestamp |
| `F.month(col)` | Extracts the month number (1–12) |
| `F.to_date(col)` | Casts a string/timestamp to DateType |
| `.alias("new_name")` | Renames the output column |

**Docs:** [PySpark Functions](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/functions.html) · [DataFrame API](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/dataframe.html)

> **Date tip:** `F.min` / `F.max` on a timestamp column return the earliest / latest timestamp.
> Wrap with `F.to_date(...)` to get just the date portion.

In [0]:
# Run this example first -- then solve the problems below.
# NOTE: this example is not a solution to any problem

df = spark.table("samples.bakehouse.sales_transactions")

# Aggregate across the whole table
df.agg(
    F.count("*").alias("num_transactions"),
    F.countDistinct("customerID").alias("unique_customers"),
    F.round(F.avg("totalPrice"), 2).alias("avg_order_value")
).show()

# Extract year and month from the dateTime column
df.select(
    F.year("dateTime").alias("yr"),
    F.month("dateTime").alias("mo")
).groupBy("yr", "mo").count().orderBy("yr", "mo").show(5)

## Problem 1

Calculate the **total number of transactions** and the **total revenue** across
all sales. Load `samples.bakehouse.sales_transactions` and return a single row.

**Expected output columns:**
- `total_transactions` - count of all transaction records
- `total_revenue` - sum of `totalPrice` across all transactions

In [0]:
df = spark.table("samples.bakehouse.sales_transactions")
display(df.head(5))

In [0]:
# Problem 1 - write your solution here
# Assign your result to: result_1

result_1 = df.agg(F.count("*").alias("total_transactions"), F.sum("totalPrice").alias("total_revenue"))  # replace this

In [0]:
# ── Tests for Problem 1 ──────────────────────────────────────────
assert result_1 is not None, "result_1 is None - did you forget to assign your DataFrame?"
assert hasattr(result_1, 'columns'), "result_1 must be a Spark DataFrame"
cols = [c.lower() for c in result_1.columns]
assert 'total_transactions' in cols, "Missing column: total_transactions"
assert 'total_revenue' in cols, "Missing column: total_revenue"
assert len(cols) == 2, f"Expected exactly 2 columns, got {len(cols)}: {cols}"
cnt = result_1.count()
assert cnt == 1, f"Expected exactly 1 row, got {cnt}"
row = result_1.collect()[0]
assert row['total_transactions'] > 0, "total_transactions must be > 0"
assert row['total_revenue'] > 0, "total_revenue must be > 0"
print(f"Problem 1 passed ✓  ({cnt} rows, transactions={row['total_transactions']}, revenue={row['total_revenue']})")

## Problem 2

List all **unique products** sold by the bakehouse, sorted alphabetically.
Each product name should appear exactly once.

**Expected output columns:**
- `product` - unique product name (sorted A → Z)

In [0]:
# Problem 2 - write your solution here
# Assign your result to: result_2

result_2 = df.select("product").distinct().orderBy(F.col("product"))

In [0]:
# ── Tests for Problem 2 ──────────────────────────────────────────
assert result_2 is not None, "result_2 is None - did you forget to assign your DataFrame?"
assert hasattr(result_2, 'columns'), "result_2 must be a Spark DataFrame"
cols = [c.lower() for c in result_2.columns]
assert 'product' in cols, "Missing column: product"
assert len(cols) == 1, f"Expected exactly 1 columns, got {len(cols)}: {cols}"
cnt = result_2.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
products = [r['product'] for r in result_2.collect()]
assert products == sorted(products), "Products must be sorted alphabetically (ascending)"
assert len(products) == len(set(products)), "Product names must be distinct"
print(f"Problem 2 passed ✓  ({cnt} rows)")

## Problem 3

Summarise transactions by **payment method**: count how many transactions
used each method and calculate the total revenue each method generated.

**Expected output columns:**
- `paymentMethod` - the payment method used
- `transaction_count` - number of transactions using that method
- `total_revenue` - total `totalPrice` for that payment method

In [0]:
# Problem 3 - write your solution here
# Assign your result to: result_3

result_3 = df.groupBy("paymentMethod").agg(
    F.count("*").alias("transaction_count"),
    F.sum("totalPrice").alias("total_revenue")
)

In [0]:
display(result_3)

In [0]:
# ── Tests for Problem 3 ──────────────────────────────────────────
assert result_3 is not None, "result_3 is None - did you forget to assign your DataFrame?"
assert hasattr(result_3, 'columns'), "result_3 must be a Spark DataFrame"
cols = [c.lower() for c in result_3.columns]
assert 'paymentmethod' in cols, "Missing column: paymentMethod"
assert 'transaction_count' in cols, "Missing column: transaction_count"
assert 'total_revenue' in cols, "Missing column: total_revenue"
assert len(cols) == 3, f"Expected exactly 3 columns, got {len(cols)}: {cols}"
cnt = result_3.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
rows = result_3.collect()
assert all(r['transaction_count'] > 0 for r in rows), "All transaction counts must be positive"
assert all(r['total_revenue'] > 0 for r in rows), "All revenue values must be positive"
print(f"Problem 3 passed ✓  ({cnt} rows)")

## Problem 4

Calculate the **average unit price** for each product, sorted so the most
expensive products appear first.

**Expected output columns:**
- `product` - product name
- `avg_unit_price` - average of `unitPrice` for that product (sorted descending)

In [0]:
# Problem 4 - write your solution here
# Assign your result to: result_4

result_4 = df.groupBy("product").agg(
    F.avg("unitPrice").alias("avg_unit_price")
).orderBy(F.col("avg_unit_price").desc())

In [0]:
# ── Tests for Problem 4 ──────────────────────────────────────────
assert result_4 is not None, "result_4 is None - did you forget to assign your DataFrame?"
assert hasattr(result_4, 'columns'), "result_4 must be a Spark DataFrame"
cols = [c.lower() for c in result_4.columns]
assert 'product' in cols, "Missing column: product"
assert 'avg_unit_price' in cols, "Missing column: avg_unit_price"
assert len(cols) == 2, f"Expected exactly 2 columns, got {len(cols)}: {cols}"
cnt = result_4.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
prices = [r['avg_unit_price'] for r in result_4.collect()]
assert prices == sorted(prices, reverse=True), "Results must be sorted by avg_unit_price descending"
assert all(p > 0 for p in prices), "All avg_unit_price values must be positive"
print(f"Problem 4 passed ✓  ({cnt} rows)")

## Problem 5

Extract the **calendar date** (without the time component) from the `dateTime`
column and count the number of transactions that occurred on each date.
Use `F.to_date` or `F.date_trunc` to strip the time.

**Expected output columns:**
- `date` - the transaction date (date type, no time)
- `transaction_count` - number of transactions on that date

In [0]:
# Problem 5 - write your solution here
# Assign your result to: result_5

result_5 = df.groupBy(
    F.to_date("dateTime").alias("date")
).agg(F.count("*").alias("transaction_count"))

In [0]:
display(result_5.head(5))

In [0]:
result_5 = df.groupBy(
    F.date_trunc("day", "dateTime").alias("date")
).agg(F.count("*").alias("transaction_count"))

In [0]:
# ── Tests for Problem 5 ──────────────────────────────────────────
assert result_5 is not None, "result_5 is None - did you forget to assign your DataFrame?"
assert hasattr(result_5, 'columns'), "result_5 must be a Spark DataFrame"
cols = [c.lower() for c in result_5.columns]
assert 'date' in cols, "Missing column: date"
assert 'transaction_count' in cols, "Missing column: transaction_count"
assert len(cols) == 2, f"Expected exactly 2 columns, got {len(cols)}: {cols}"
cnt = result_5.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
rows = result_5.collect()
assert all(r['transaction_count'] > 0 for r in rows), "All transaction_count values must be positive"
print(f"Problem 5 passed ✓  ({cnt} rows)")

## Problem 6

Find the **date range** of all transactions — the very first transaction date
and the most recent transaction date in the dataset.
Return a **single row** with both values as plain dates (no time component).

**Expected output columns:**
- `first_transaction_date` — earliest date any transaction was recorded
- `latest_transaction_date` — most recent date any transaction was recorded

**Hint:** use `F.min` and `F.max` on `dateTime`, then `F.to_date` to strip the time.

In [0]:
# Problem 6 - write your solution here
# Assign your result to: result_6

result_6 = df.agg(
    F.to_date(F.min("dateTime")).alias("first_transaction_date"),
    F.to_date(F.max("dateTime")).alias("latest_transaction_date")
)

In [0]:
display(result_6)

In [0]:
# ── Tests for Problem 6 ──────────────────────────────────────────
from pyspark.sql.types import DateType
assert result_6 is not None, "result_6 is None - did you forget to assign your DataFrame?"
assert hasattr(result_6, 'columns'), "result_6 must be a Spark DataFrame"
cols = [c.lower() for c in result_6.columns]
assert 'first_transaction_date' in cols, "Missing column: first_transaction_date"
assert 'latest_transaction_date' in cols, "Missing column: latest_transaction_date"
assert len(cols) == 2, f"Expected exactly 2 columns, got {len(cols)}: {cols}"
cnt = result_6.count()
assert cnt == 1, f"Expected exactly 1 row, got {cnt}"
row = result_6.collect()[0]
assert row['first_transaction_date'] is not None, "first_transaction_date must not be null"
assert row['latest_transaction_date'] is not None, "latest_transaction_date must not be null"
assert row['first_transaction_date'] <= row['latest_transaction_date'], \
    "first_transaction_date must be on or before latest_transaction_date"
print(f"Problem 6 passed ✓  first={row['first_transaction_date']}, latest={row['latest_transaction_date']}")

## Problem 7

For each **customer**, find their **most recent order date** — the last time they placed
a transaction. Sort the results so the customers who ordered most recently appear first.

This lets you answer questions like *"which customers are still active?"* or
*"who placed the very first order in the dataset?"* (sort ascending instead).

**Expected output columns:**
- `customerID` — customer identifier
- `last_order_date` — date of their most recent transaction (date only, no time)

**Sorted:** by `last_order_date` descending (most recent customers first).

**Docs:** [groupBy + agg](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.groupBy.html) · [F.max](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.max.html)

In [0]:
# Problem 7 - write your solution here
# Assign your result to: result_7

result_7 = df.groupBy("customerID").agg(
    F.to_date(F.max("dateTime")).alias("last_order_date")
).orderBy(F.col("last_order_date").desc())

In [0]:
# ── Tests for Problem 7 ──────────────────────────────────────────
assert result_7 is not None, "result_7 is None - did you forget to assign your DataFrame?"
assert hasattr(result_7, 'columns'), "result_7 must be a Spark DataFrame"
cols = [c.lower() for c in result_7.columns]
assert 'customerid' in cols, "Missing column: customerID"
assert 'last_order_date' in cols, "Missing column: last_order_date"
assert len(cols) == 2, f"Expected exactly 2 columns, got {len(cols)}: {cols}"
cnt = result_7.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
# Verify sorted descending
dates = [r['last_order_date'] for r in result_7.collect()]
assert dates == sorted(dates, reverse=True), "Results must be sorted by last_order_date descending"
# Each customerID should appear once
ids = [r['customerID'] for r in result_7.collect()]
assert len(ids) == len(set(ids)), "Each customerID must appear exactly once"
print(f"Problem 7 passed ✓  ({cnt} customers, most recent={dates[0]}, earliest first-order={dates[-1]})")